# 📘 Notebook 14: Convolutional Neural Networks (CNNs)

### Course: *From Python to Deep Learning & Fine-Tuning*
**Instructor:** Ioannis Tsioulis

*Module D: Deep Learning · Notebook 3 of 4 in this module · (14 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why fully-connected networks struggle with images
- Understand the **convolution** operation and what a **filter/kernel** detects
- Describe **pooling** and why it helps
- Outline a typical **CNN architecture** and the idea of learned feature hierarchies
- Build and train a small CNN in PyTorch on image data

> **Prerequisites:** Notebooks 12-13 (neurons, training, PyTorch).
>
> 🔭 **Why CNNs matter.** Convolutional networks are the architecture that made modern computer vision possible, including medical imaging, where a CNN can learn to read an X-ray or an MRI scan. They introduce one powerful new idea on top of everything you already know.

> ℹ️ **Setup note.** `import piplite; await piplite.install(['torch','torchvision','numpy','matplotlib'])`. `torchvision` and the dataset download may be heavy in a browser kernel; if so, read through and run the code in a standard Python environment. The concepts below stand on their own.

## 1. Why fully-connected networks struggle with images

### The problem of scale
Consider a modest 200×200 colour image: that is 200 × 200 × 3 = 120,000 input numbers. A single fully-connected hidden layer with 1,000 neurons would need 120 **million** weights, for one layer. This is wasteful and quickly becomes untrainable.

### The problem of structure
Worse, a fully-connected layer treats every pixel independently and ignores **spatial structure**. But in images, nearby pixels are highly related, and a cat is a cat whether it appears in the top-left or bottom-right. Flattening the image throws this away. We need an architecture that respects locality and reuses what it learns across the image.

## 2. The convolution operation

### Intuition first
A **convolution** slides a small window, a **filter** (or *kernel*), across the image. At each position it computes a weighted sum of the pixels under it (a dot product, the operation from Notebook 05 again). The result is a **feature map** highlighting where in the image that filter's pattern appears.

### What a filter detects
Different filters detect different things: one might respond to vertical edges, another to horizontal edges, another to a particular texture. Crucially, in a CNN these filters are **learned** by gradient descent, the network discovers for itself which patterns are useful. Let us apply a hand-made edge-detecting filter to see the effect:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A simple synthetic image: a bright square on a dark background
image = np.zeros((20, 20))
image[5:15, 5:15] = 1.0

# A vertical-edge-detecting filter (Sobel-like)
kernel = np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]])

def convolve2d(img, k):
    kh, kw = k.shape
    out = np.zeros((img.shape[0]-kh+1, img.shape[1]-kw+1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(img[i:i+kh, j:j+kw] * k)   # dot product under the window
    return out

edges = convolve2d(image, kernel)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(image, cmap="gray"); axes[0].set_title("Original")
axes[1].imshow(edges, cmap="gray"); axes[1].set_title("After edge filter")
plt.tight_layout(); plt.show()

The filter lit up exactly at the **vertical edges** of the square. A CNN stacks many such filters and *learns* their values, rather than us hand-designing them.

> 🧠 **Parameter sharing, the key efficiency.** The same small filter (here just 9 numbers) is reused across the entire image. So the network learns 'what a vertical edge looks like' **once** and applies it everywhere. This is why a convolutional layer needs vastly fewer weights than a fully-connected one, and why it naturally handles objects appearing in different positions.

## 3. Pooling: summarising and shrinking

### The idea
After convolution, a **pooling** layer shrinks each feature map by summarising small regions, most commonly **max pooling**, which keeps only the largest value in each little window. This reduces computation, and makes the representation more robust to small shifts in the input (a feature detected slightly off-position still survives).

In [ ]:
def max_pool2d(img, size=2):
    h, w = img.shape
    out = np.zeros((h//size, w//size))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            region = img[i*size:(i+1)*size, j*size:(j+1)*size]
            out[i, j] = region.max()
    return out

pooled = max_pool2d(image, size=2)
print("Original shape:", image.shape, "-> pooled shape:", pooled.shape)

## 4. A typical CNN architecture

A CNN stacks these building blocks into a pipeline:

```
Input image
  -> [Convolution + ReLU] -> [Pooling]      (detect simple features: edges)
  -> [Convolution + ReLU] -> [Pooling]      (combine into complex features: shapes, parts)
  -> Flatten
  -> Fully-connected layer(s)               (reasoning over the features)
  -> Output (class probabilities)
```

### The feature hierarchy
This layered design produces a **hierarchy of features**: early layers detect simple things (edges, colours), middle layers combine them into parts (an eye, a wheel), and later layers assemble whole objects (a face, a car). The network learns this hierarchy automatically. Famous architectures, **LeNet** (1998), **VGG**, **ResNet**, are all variations on this theme, differing mainly in depth and clever connection patterns.

## 5. A small CNN in PyTorch

Here is a compact CNN for classifying small images (e.g. handwritten digits). Notice that it is the same `nn.Module` class structure from Notebook 13: we have simply swapped some `nn.Linear` layers for `nn.Conv2d` and `nn.MaxPool2d`:

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)   # 1 channel in, 8 filters
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)  # 8 -> 16 filters
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc    = nn.Linear(16 * 7 * 7, n_classes)            # for 28x28 inputs

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))   # 28x28 -> 14x14
        x = self.pool(torch.relu(self.conv2(x)))   # 14x14 -> 7x7
        x = x.flatten(start_dim=1)                 # flatten for the dense layer
        x = self.fc(x)
        return x

model = SmallCNN()
print(model)

# Count parameters -- far fewer than an equivalent fully-connected net would need
n_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {n_params:,}")

The **training loop is identical** to Notebook 13: forward pass, compute loss (cross-entropy for multi-class), `backward()`, `optimizer.step()`. The only thing that changed is the architecture inside `forward`. This is the payoff of understanding the fundamentals, new architectures are new *forward passes*, trained by the same loop.

> ⚠️ **Common pitfalls**
>
> - Getting the flattened size wrong (`16 * 7 * 7` here) is the classic CNN bug. Trace the spatial size through each pool layer: two 2×2 poolings turn 28×28 into 7×7. Printing `x.shape` inside `forward` while debugging saves much frustration.
> - Images must have a channel dimension: PyTorch expects shape `(batch, channels, height, width)`.

---
## ✏️ Exercises

### Exercise 1
Create a 3×3 filter that detects **horizontal** edges (the transpose idea of the vertical one used above) and apply your `convolve2d` to the `image`. How does the output differ from the vertical-edge result?

In [ ]:
import numpy as np
# (Re-use convolve2d and image from earlier.)
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
kernel_h = np.array([[-1, -2, -1],
                     [ 0,  0,  0],
                     [ 1,  2,  1]])
edges_h = convolve2d(image, kernel_h)
# This lights up the TOP and BOTTOM edges of the square, instead of the sides.
```

*Rotating the filter rotates which orientation of edge it detects. A CNN learns a whole bank of such filters automatically.*
</details>

### Exercise 2
Reason about parameter counts. For a 28×28 grayscale image, how many weights would a single fully-connected layer with 100 neurons need (ignoring biases)? Compare conceptually with `conv1` above (8 filters of size 3×3).

In [ ]:
# Your calculation here:


<details><summary>💡 Show solution</summary>

```python
fc_weights = 28 * 28 * 100      # = 78,400 weights for ONE dense layer
conv_weights = 8 * (3 * 3)      # = 72 weights for the convolutional layer
print(fc_weights, conv_weights)
```

*Roughly 78,400 vs 72. Parameter sharing makes convolution dramatically more efficient, and it also respects image structure, which the dense layer ignores.*
</details>

## 🔑 Key takeaways

- Fully-connected layers are wasteful on images and ignore spatial structure.
- **Convolution** slides a learned **filter** over the image to build **feature maps** detecting patterns.
- **Parameter sharing** makes CNNs efficient and naturally tolerant of an object's position.
- **Pooling** shrinks feature maps and adds robustness to small shifts.
- A CNN learns a **hierarchy of features**; architectures like LeNet/VGG/ResNet build on this.
- In PyTorch a CNN is the same `nn.Module` pattern with `Conv2d`/`MaxPool2d`, trained by the identical loop.

---
**Next:** *Notebook 15: Sequence Models & the Road to Attention*, where we handle text and time series and motivate the Transformer.

---
*© Ioannis Tsioulis. *From Python to Deep Learning & Fine-Tuning*. Prepared for educational use.*